# Entraînement LoRA Flux.1 Schnell — personnage Solle

Notebook **Kaggle** (GPU **T4 x2**, 2×16 Go) pour entraîner un LoRA **Flux.1 Schnell** sur Solle.

**Pourquoi Schnell :** licence **Apache 2.0** = usage commercial libre, génération rapide (4 steps).
Avantages vs SDXL : texte lisible dans les images, styles réaliste/cartoon à la demande sans tags booru.

> ⚠️ Le repo HF de Schnell est **gated** (barrière d'accès) : il faut un token HF pour le télécharger.
> La **licence reste Apache 2.0** — le token est juste une formalité de download, pas une restriction d'usage commercial.
> L'entraînement utilise aussi l'adaptateur officiel `ostris/FLUX.1-schnell-training-adapter`.

> 🔑 **Flux ne tient pas sur un seul T4 16 Go** (transformer 12 Go + T5 9,5 Go = ~21 Go).
> Ce notebook utilise `split_model_over_gpus` pour **répartir le modèle sur les 2 T4** → il FAUT l'accélérateur **T4 x2**.

## Prérequis AVANT de lancer

### 1. Token HuggingFace (OBLIGATOIRE)
- Va sur https://huggingface.co/black-forest-labs/FLUX.1-schnell → clique **Agree and access repository**
- Crée un token : https://huggingface.co/settings/tokens (type = **Read**)
- Dans Kaggle → ton profil → **Settings → Secrets** → **Add New Secret**
  - Name : `HF_TOKEN`
  - Value : ton token (commence par `hf_...`)
  - ⚠️ Attache bien le secret à CE notebook (case à cocher dans le panneau Secrets)

### 2. Dataset NETTOYÉ
- Utilise le dataset **sans watermark Gemini** (script `remove_gemini_watermark.py` déjà passé).
- Crée un Dataset Kaggle privé avec les 79 PNG + 79 TXT de `training/dataset/`.
- Menu de droite → **Add Input** → ajoute le dataset (peu importe le nom, la cellule 4 scanne tout `/kaggle/input/`).

### 3. Config GPU
- **Accelerator = GPU T4 x2** (OBLIGATOIRE — le modèle est réparti sur les 2 GPU)
- **Internet = ON** (indispensable)
- **Persistence = Files only** (garde le cache si tu relances)

### Durée estimée
- Installation + téléchargement Flux : ~20-30 min
- Entraînement 1500 steps : ~3-5h sur T4
- **Lance en mode Commit** (Save & Run All) pour que ça tourne sans garder l'onglet ouvert

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi

## 2. Installer ai-toolkit (moteur d'entraînement Flux)

**ai-toolkit** (ostris) est l'outil de référence pour entraîner des LoRA Flux.

In [ ]:
%cd /kaggle/working
!git clone https://github.com/ostris/ai-toolkit
%cd ai-toolkit
!git submodule update --init --recursive
!pip install -q -r requirements.txt
# bitsandbytes pour la quantisation 8-bit (indispensable sur T4 16 Go)
!pip install -q bitsandbytes>=0.43.0
print('Installation terminée.')

## 3. Connexion HuggingFace — OPTIONNEL

Schnell est public : cette cellule n'est **pas obligatoire**. Elle se connecte seulement si tu as
défini un secret `HF_TOKEN` dans Kaggle. Sinon elle passe sans erreur.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Schnell est GATED sur HF : token obligatoire.
try:
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    hf_token = None

if not hf_token:
    raise RuntimeError(
        "HF_TOKEN manquant ! Flux Schnell est gated.\n"
        "1) Accepte l'acces : https://huggingface.co/black-forest-labs/FLUX.1-schnell\n"
        "2) Cree un token Read : https://huggingface.co/settings/tokens\n"
        "3) Kaggle > Settings > Secrets > Add 'HF_TOKEN' et attache-le a ce notebook."
    )

os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token  # certaines libs lisent ce nom
login(token=hf_token, add_to_git_credential=False)
print('Connecté à HuggingFace — token OK.')

## 4. Préparer le dataset

ai-toolkit attend un dossier plat avec images + `.txt` côte à côte.
Pas de structure `repeats_name/` comme kohya — les steps contrôlent tout.

In [ ]:
import os, glob, shutil

# Scanne TOUT /kaggle/input/ — robuste quel que soit le nom/slug du dataset.
INPUT_ROOT = '/kaggle/input'
DST = '/kaggle/working/train_data'
os.makedirs(DST, exist_ok=True)

# Liste d'abord ce qui est monté pour diagnostic
print('Datasets montés sous /kaggle/input/ :')
for d in sorted(glob.glob(f'{INPUT_ROOT}/*')):
    print('  -', d)

found = []
for ext in ('png', 'jpg', 'jpeg', 'txt'):
    found += glob.glob(f'{INPUT_ROOT}/**/*.{ext}', recursive=True)
found = sorted(set(found))

for f in found:
    shutil.copy(f, DST)

imgs = len(glob.glob(DST + '/*.png')) + len(glob.glob(DST + '/*.jpg')) + len(glob.glob(DST + '/*.jpeg'))
txts = len(glob.glob(DST + '/*.txt'))
print(f'\nFichiers copiés : images={imgs} | captions={txts}')
assert imgs > 0, 'AUCUNE image trouvee ! Verifie que le dataset est bien ajoute via Add Input (panneau de droite).'
assert imgs == txts, f'Mismatch ! {imgs} images vs {txts} captions'
print('Exemple caption :', open(sorted(glob.glob(DST + '/*.txt'))[0]).read())

## 5. Créer la configuration d'entraînement (Schnell)

Points clés :
- `name_or_path: FLUX.1-schnell` + `assistant_lora_path` → adaptateur d'entraînement Schnell (stabilise l'apprentissage)
- `is_flux: true` + `quantize: true` → charge Flux en 8-bit pour tenir en 16 Go VRAM
- `dtype: bf16` → plus stable que fp16 pour Flux
- `train_text_encoder: false` → entraîne seulement le transformer (recommandé pour personnage)
- `steps: 1500` → bon équilibre apprentissage/sur-entraînement sur 79 images
- `sample_steps: 4` + `guidance_scale: 1` → Schnell est distillé (4 steps suffisent)
- Le trigger `sollechar` est appris depuis les captions (déjà en 1er tag partout)

In [ ]:
import os, yaml

config = {
    'job': 'extension',
    'config': {
        'name': 'solle_flux',
        'process': [{
            'type': 'sd_trainer',
            'training_folder': '/kaggle/working/output',
            'performance_log_every': 100,
            'device': 'cuda:0',

            'network': {
                'type': 'lora',
                'linear': 16,
                'linear_alpha': 16
            },

            'save': {
                'dtype': 'float16',
                'save_every': 500,
                'max_step_saves_to_keep': 4
            },

            'datasets': [{
                'folder_path': '/kaggle/working/train_data',
                'caption_ext': '.txt',
                # 0 obligatoire avec cache_text_embeddings (bug dropout/cache #374)
                'caption_dropout_rate': 0.0,
                'shuffle_tokens': False,
                'cache_latents_to_disk': True,
                'cache_text_embeddings': True,
                # 16 Go par GPU : on reste en 512
                'resolution': [512]
            }],

            'train': {
                'batch_size': 1,
                'steps': 1500,
                'gradient_accumulation_steps': 1,
                'train_unet': True,
                'train_text_encoder': False,
                # Decharge le T5 du VRAM apres le caching (libere cuda:0 pour le training)
                'unload_text_encoder': True,
                'gradient_checkpointing': True,
                'noise_scheduler': 'flowmatch',
                'optimizer': 'adamw8bit',
                'lr': 1e-4,
                'ema_config': {
                    'use_ema': True,
                    'ema_decay': 0.99
                },
                'dtype': 'bf16'
            },

            'model': {
                'name_or_path': 'black-forest-labs/FLUX.1-schnell',
                'assistant_lora_path': 'ostris/FLUX.1-schnell-training-adapter',
                'is_flux': True,
                'quantize': True,
                # low_vram : garde le transformer sur CPU pendant la quantisation (sinon OOM)
                'low_vram': True,
                # MULTI-GPU : repartit le transformer sur les 2x T4
                'split_model_over_gpus': True,
                # scale 2.0 : pousse ~tout le transformer sur cuda:1, laisse cuda:0 quasi libre
                # pour le forward T5 du caching (le forward quanto deborde a scale 1.0)
                'split_model_other_module_param_count_scale': 2.0
            },

            'sample': {
                'sampler': 'flowmatch',
                'sample_every': 500,
                'width': 1024,
                'height': 1024,
                'prompts': [
                    'sollechar, comic illustration style, full body, standing, looking at viewer',
                    'sollechar standing in London, red phone booth, holding a sign that says BUY SOL, photorealistic background',
                    'sollechar, cartoon style, skateboarding in New York, dynamic pose, urban street'
                ],
                'neg': '',
                'seed': 42,
                'walk_seed': True,
                'guidance_scale': 1,
                'sample_steps': 4
            }
        }]
    },
    'meta': {
        'name': '[name]',
        'version': '1.0'
    }
}

os.makedirs('/kaggle/working/configs', exist_ok=True)
cfg_path = '/kaggle/working/configs/solle_flux.yaml'
with open(cfg_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print('Config écrite :')
print(open(cfg_path).read())

## 6. Lancer l'entraînement

Une seule commande — ai-toolkit gère tout (téléchargement Flux + adaptateur, cache latents, sampling).

**Logs à surveiller :**
- `loss: 0.4xxx` → normal en début (Flux démarre plus haut que SDXL)
- `loss: 0.2-0.3` → bonne convergence vers step 500-1000
- Images de sample générées automatiquement toutes les 500 steps dans `output/samples/`
- **Regarde le sample du prompt 'BUY SOL'** : si le texte est lisible, c'est gagné
- Checkpoints sauvegardés toutes les 500 steps dans `output/`

In [ ]:
%cd /kaggle/working/ai-toolkit
# expandable_segments réduit la fragmentation VRAM (aide sur 16 Go)
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python run.py /kaggle/working/configs/solle_flux.yaml

## 7. Vérifier et télécharger le LoRA

Les fichiers sont dans `/kaggle/working/output/`.
Télécharge via l'onglet **Output** (panneau de droite dans Kaggle).

**Fichiers à garder :**
- `solle_flux.safetensors` → version finale (step 1500)
- `solle_flux-000001000.safetensors` → checkpoint step 1000 (backup si sur-entraînement)
- `output/samples/` → images générées automatiquement (pour juger la qualité)

**Usage à l'inférence (Schnell, ex. HuggingFace Spaces) :**
```python
import torch
from diffusers import FluxPipeline

pipe = FluxPipeline.from_pretrained(
    'black-forest-labs/FLUX.1-schnell', torch_dtype=torch.bfloat16
)
pipe.load_lora_weights('solle_flux.safetensors')
pipe.enable_model_cpu_offload()

image = pipe(
    'sollechar standing in London, holding a sign that says BUY SOL, red phone booth',
    num_inference_steps=4,      # Schnell = 4 steps
    guidance_scale=0.0,         # Schnell ignore la guidance
    joint_attention_kwargs={'scale': 0.8},  # force du LoRA (0.7-0.9)
).images[0]
image.save('solle_london.png')
```

In [ ]:
import os, glob

output_dir = '/kaggle/working/output'
loras = sorted(glob.glob(f'{output_dir}/**/*.safetensors', recursive=True))
samples = sorted(glob.glob(f'{output_dir}/**/samples/*.png', recursive=True))

print(f'LoRA sauvegardés ({len(loras)}) :')
for f in loras:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'  {os.path.basename(f)} — {size_mb:.1f} Mo')

print(f'\nImages de sample ({len(samples)}) :')
for f in samples[-6:]:  # les 6 dernières
    print(f'  {os.path.basename(f)}')